# c8 — Viết lại câu hỏi + thuật ngữ trên Google Colab (một phần của k8)

**Runtime → Change runtime type → T4 GPU.** Mỗi phiên Colab 1 GPU → chạy một phần (`SHARD`), các phần khác ở Kaggle
(k8, `SHARDS = ['0/3', '1/3']`).
**Chuẩn bị trên Google Drive** thư mục `MyDrive/mir2/`:
- `mir2_code.zip` (CÙNG bản với Kaggle — ô (1) in VERSION để đối chiếu);
- `qtrans/queries_vi.parquet`, `queries_en.parquet`, `queries_zh.parquet` (tải từ dataset `mir2-runs`, thư mục `qtrans/`).

Kết quả ghi thẳng vào Drive: `MyDrive/mir2/work/qexp/rw3-qwen25_14b/records.jsonl`, nối sau MỖI lô → mất kết
nối thì chạy lại mọi ô, lượt đã có được bỏ qua. Xong: tải file đó về.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA.
SHARD = '0/1'                                  # cả 1.200 câu; chia với Kaggle: '2/3' (k8 chạy 0/3, 1/3)
MODEL, BATCH, QUANT = 'qwen25_14b', 32, 'fp16'  # GPU 96 GB (G4); T4: 8, '4bit'
DRIVE = '/content/drive/MyDrive/mir2'
import os, shutil, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
CODE, CFG = '/content/code', '/content/config.yaml'
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(f'{DRIVE}/mir2_code.zip').extractall(CODE)
print('mir2-code', open(f'{CODE}/VERSION.txt', encoding='utf-8').read().strip())
miss = [lg for lg in ('vi', 'en', 'zh') if not os.path.exists(f'{DRIVE}/qtrans/queries_{lg}.parquet')]
assert not miss, f'Thiếu {DRIVE}/qtrans/queries_{miss}.parquet'
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import write_config
write_config(f'{CODE}/configs/retrieval.yaml', CFG, {'translate.out_dir': f'{DRIVE}/qtrans', 'paths.work_dir': f'{DRIVE}/work'})
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U bitsandbytes accelerate

In [ ]:
# (2) Sinh: G4 fp16 ước ~20–50 phút cho 1.200 câu × 3 ngôn ngữ (T4 4-bit: ~4 s/lượt), chưa kể ~10 phút tải + nạp mô hình
!python -m mirret.query_expand generate --config {CFG} --model {MODEL} --shard {SHARD} --batch {BATCH} --quant {QUANT}

In [ ]:
# (3) Kiểm tra nhanh: số lượt, lỗi JSON, 3 câu mẫu → tải records-*.jsonl trong Drive về
!python -m mirret.query_expand show --config {CFG} --model {MODEL} --limit 3
!ls -la {DRIVE}/work/qexp/rw3-qwen25_14b/